In [0]:
# ============================================
# GOLD LAYER - CLIENT CHURN + RFM
# ============================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window

fact = spark.table("gold.fact_transactions")
dim_date = spark.table("gold.dim_date")

In [0]:
# Join to get real date

fact_with_date = (
    fact.join(
        dim_date,
        "date_sk",
        "left"
    )
)

In [0]:
# Basic client metrics

client_metrics = (
    fact_with_date
    .groupBy("client_sk")
    .agg(
        min("date").alias("first_transaction_date"),
        max("date").alias("last_transaction_date"),
        count("*").alias("frequency"),
        sum("amount").alias("monetary")
    )
)

In [0]:
client_metrics = client_metrics.withColumn(
    "avg_transaction_value",
    col("monetary") / col("frequency")
)

In [0]:
max_date = fact_with_date.agg(max("date")).collect()[0][0]
print("Max dataset date:", max_date)

Max dataset date: 2019-10-31


In [0]:
# Recency + churn

client_metrics = (
    client_metrics
    .withColumn(
        "recency_days",
        datediff(lit(max_date), col("last_transaction_date"))
    )
    .withColumn(
        "is_churned",
        when(col("recency_days") > 90, True).otherwise(False)
    )
)

In [0]:
client_metrics = client_metrics.withColumn(
    "churn_reason",
    when(col("recency_days") > 180, "Inactive > 6 months")
    .when(col("recency_days") > 90, "Inactive > 3 months")
    .otherwise("Active")
)

In [0]:
# Percentile thresholds (approximate for scalability)

recency_q = client_metrics.approxQuantile(
    "recency_days",
    [0.2, 0.4, 0.6, 0.8],
    0.01
)

frequency_q = client_metrics.approxQuantile(
    "frequency",
    [0.2, 0.4, 0.6, 0.8],
    0.01
)

monetary_q = client_metrics.approxQuantile(
    "monetary",
    [0.2, 0.4, 0.6, 0.8],
    0.01
)

print("Recency thresholds:", recency_q)
print("Frequency thresholds:", frequency_q)
print("Monetary thresholds:", monetary_q)

Recency thresholds: [0.0, 0.0, 0.0, 0.0]
Frequency thresholds: [6149.0, 8252.0, 10537.0, 13709.0]
Monetary thresholds: [280562.0, 384500.09, 502242.35, 706232.81]


In [0]:
r1, r2, r3, r4 = recency_q
f1, f2, f3, f4 = frequency_q
m1, m2, m3, m4 = monetary_q

In [0]:
# Recency
client_metrics = client_metrics.withColumn(
    "R_score",
    when(col("recency_days") <= r1, 5)
    .when(col("recency_days") <= r2, 4)
    .when(col("recency_days") <= r3, 3)
    .when(col("recency_days") <= r4, 2)
    .otherwise(1)
)

In [0]:
# Frequency
client_metrics = client_metrics.withColumn(
    "F_score",
    when(col("frequency") <= f1, 1)
    .when(col("frequency") <= f2, 2)
    .when(col("frequency") <= f3, 3)
    .when(col("frequency") <= f4, 4)
    .otherwise(5)
)

In [0]:
# Monetary

client_metrics = client_metrics.withColumn(
    "M_score",
    when(col("monetary") <= m1, 1)
    .when(col("monetary") <= m2, 2)
    .when(col("monetary") <= m3, 3)
    .when(col("monetary") <= m4, 4)
    .otherwise(5)
)

In [0]:
# Final RFM

client_metrics = client_metrics.withColumn(
    "RFM_score",
    concat(col("R_score"), col("F_score"), col("M_score"))
)

In [0]:
client_metrics = client_metrics.withColumn(
    "segment",
    when(col("RFM_score") == "555", "Champions")
    .when((col("R_score") >= 4) & (col("F_score") >= 4), "Loyal")
    .when((col("R_score") <= 2) & (col("F_score") >= 3), "At Risk")
    .when((col("R_score") <= 2) & (col("F_score") <= 2), "Lost")
    .otherwise("Potential")
)

In [0]:
client_metrics = client_metrics.withColumn(
    "rf_segment",
    concat(col("R_score"), col("F_score"))
)

In [0]:
client_metrics = client_metrics.withColumn(
    "customer_lifetime_value",
    col("avg_transaction_value") * col("frequency")
)

In [0]:
client_metrics = client_metrics.withColumn(
    "rf_segment_label",
    when((col("R_score") == 5) & (col("F_score") >= 4), "Champions")
    .when((col("R_score") >= 4) & (col("F_score") >= 3), "Loyal Customers")
    .when((col("R_score") >= 4) & (col("F_score") <= 2), "Potential Loyalists")
    .when((col("R_score") == 3) & (col("F_score") >= 3), "Need Attention")
    .when((col("R_score") == 2) & (col("F_score") >= 3), "At Risk")
    .when((col("R_score") == 1) & (col("F_score") >= 4), "Cannot Lose Them")
    .when((col("R_score") == 1) & (col("F_score") <= 3), "Lost Customers")
    .otherwise("Others")
)

In [0]:
client_metrics.groupBy("rf_segment_label") \
    .count() \
    .orderBy(col("count").desc()) \
    .show()

+-------------------+-----+
|   rf_segment_label|count|
+-------------------+-----+
|          Champions|  483|
|Potential Loyalists|  395|
|    Loyal Customers|  231|
|     Lost Customers|  102|
|   Cannot Lose Them|    8|
+-------------------+-----+



In [0]:
client_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.client_rfm_metrics")